# 임시 진단: Ollama · LangChain · Structured Output 호출 분리

이 노트북은 14번의 `하닉` 사전 호출이 오래 걸린 원인을 분리하기 위한 **임시 진단용** 파일이다. 성능 비교·최종 구조 선택·결과 보고서 작성 목적이 아니며, 진단 후 삭제할 수 있다.

> 현재 사용자가 GPU에 올린 Ollama runner를 그대로 사용한다. `num_ctx`, GPU, keep-alive, 출력 길이 같은 runner 옵션은 지정하지 않는다.

## 실행 순서

아래 세 호출은 **각각 별도 셀**이다. 앞 셀이 정상 완료된 경우에만 다음 셀을 실행한다. 어느 셀에서 오래 멈추는지와 마지막 출력 문구를 기록한다.

1. Ollama 직접 chat 호출: 모델·Ollama 기본 경로
2. LangChain `ChatOllama` 일반 호출: LangChain 연결 경로
3. LangChain structured output 호출: 14번과 같은 구조화 출력 경로

`temperature=0`은 14번과 동일한 생성 조건을 맞추기 위한 요청 단위 옵션이며, runner 적재 설정은 바꾸지 않는다. HTTP 대기 제한은 120초다. timeout이 나면 다음 셀을 실행하지 않고, Ollama 요청이 남아 있는지 먼저 확인한다.

In [ ]:
import json
import os
from pathlib import Path
import time
from typing import Literal
import urllib.request

from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE = 0

print(f'모델: {MODEL_NAME}')
print(f'Ollama URL: {OLLAMA_BASE_URL}')
print(f'temperature: {TEMPERATURE}')
print('runner 옵션: 지정하지 않음')

DIAGNOSTIC_MESSAGES = [
    ('system', '짧고 정확하게 답하세요.'),
    ('human', '하닉은 어떤 회사의 약칭으로 해석할 수 있나요?'),
]

## 1. Ollama 직접 chat 호출

LangChain 없이 Ollama `/api/chat`을 직접 호출한다. 2번과 동일한 메시지를 사용한다. 이 셀이 오래 멈추면 LangChain 이전의 모델·Ollama 실행 경로부터 확인해야 한다.

In [ ]:
payload = {
    'model': MODEL_NAME,
    'messages': [
        {'role': role, 'content': content}
        for role, content in DIAGNOSTIC_MESSAGES
    ],
    'stream': False,
    'options': {'temperature': TEMPERATURE},
}
request = urllib.request.Request(
    f'{OLLAMA_BASE_URL}/api/chat',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
)

print('[1/3] Ollama 직접 chat 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    with urllib.request.urlopen(request, timeout=120) as response:
        direct_result = json.load(response)
    print(f'[1/3] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('content:', repr(direct_result.get('message', {}).get('content')))
    print('thinking:', repr(direct_result.get('message', {}).get('thinking')))
    print('생성 토큰 수:', direct_result.get('eval_count'))
except Exception as error:
    print(f'[1/3] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

## 2. LangChain `ChatOllama` 일반 호출

동일 모델을 LangChain으로 부르되 structured output은 사용하지 않는다. 1번은 빠르고 이 셀만 오래 멈추면 LangChain 연결·메시지 변환 경로를 우선 의심한다.

In [ ]:
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    client_kwargs={'timeout': 120},
)

print('[2/3] LangChain 일반 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    plain_response = llm.invoke(DIAGNOSTIC_MESSAGES)
    print(f'[2/3] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('content:', repr(plain_response.content))
    print('thinking:', repr(plain_response.additional_kwargs.get('thinking')))
except Exception as error:
    print(f'[2/3] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

## 3. LangChain structured output 호출

14번의 사전 호출과 같은 형식이다. 이 셀만 오래 멈추면, 모델 자체보다 structured output 형식 또는 해당 프롬프트와의 상호작용을 원인 후보로 좁힌다.

In [ ]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(
        description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태'
    )
    search_terms: list[str] = Field(
        default_factory=list,
        max_length=2,
        description='OpenDART에서 검증할 정식 기업명 검색 가설'
    )

SYSTEM_PROMPT = '''
당신은 한국 기업 리서치 시스템의 기업명 검색 계획기다.
원문이 명백한 약칭, 한글·영문 표기 차이, 또는 가벼운 철자 오타로 보이면 decision='candidate'와 함께
정식 기업명으로 보이는 검색 가설을 최대 2개 제안한다.
원문이 포괄적인 그룹명, 여러 기업을 뜻할 수 있는 표현, 또는 실제 기업으로 해석할 근거가 없는 표현이면
decision='ambiguous' 또는 decision='unknown'을 반환하고 search_terms=[]로 둔다.
corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
'''

structured_llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    client_kwargs={'timeout': 120},
)
planner = structured_llm.with_structured_output(EntitySearchPlan, include_raw=True)

print('[3/3] LangChain structured output 호출 시작', flush=True)
started_at = time.perf_counter()
try:
    structured_result = planner.invoke([
        ('system', SYSTEM_PROMPT),
        ('human', '사용자 질문: 하닉 실적을 정리해줘.\n원문 기업 표현: 하닉'),
    ])
    print(f'[3/3] 완료 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print('파싱 결과:', structured_result['parsed'])
    print('파싱 오류:', structured_result['parsing_error'])
    print('원문 응답:', structured_result['raw'])
except Exception as error:
    print(f'[3/3] 실패 / {time.perf_counter() - started_at:.2f}초', flush=True)
    print(f'{type(error).__name__}: {error}')

## 해석 기준

- 1번부터 오래 멈춤: Ollama·모델 실행 경로 또는 현재 runner 상태가 원인 후보
- 1번 정상, 2번부터 오래 멈춤: LangChain `ChatOllama` 연결·메시지 경로가 원인 후보
- 1·2번 정상, 3번만 오래 멈춤: structured output 형식·프롬프트 상호작용이 원인 후보
- 세 셀 모두 정상: 14번의 전체 노트북 상태·실행 순서·일시적 runner 상태를 별도로 확인

이 결과는 원인 후보를 좁히는 용도다. 단일 실행만으로 최종 모델·프롬프트·구조를 채택하거나 미채택하지 않는다.